# Attempt 01

In [ ]:
# Cell G1 — setup. Peak RAM ~0.
# Gold-concept subtask (separate from KB preprocessing). Read-only inputs:
#   META/MRFILES.RRF, META/MRXNS_ENG.RRF                     (UMLS 2026AA)
#   reference_data/MRCONSO.RRF, MRSTY.RRF, SemGroups.txt     (general clinical English UMLS, NOT the chest KB)
#   MedLayBench-V chest validation/test parquet              (expert + lay captions)
# Gold concepts never look at the KB, so retrieval recall and the KB ceiling are not circular.
import csv, gc, random, re, time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS   # scikit-learn's 318-word list

UMLS_META_DIR = Path(r"D:\Project\Radiology_RAG\data\raw_data\umls\umls_2026AA_full\2026AA\META")
REF_OUT  = Path(r"D:\Project\Radiology_RAG\data\subtask_data_coverage\reference_data")
GOLD_OUT = Path(r"D:\Project\Radiology_RAG\data\subtask_gold_concepts_det")
GOLD_OUT.mkdir(parents=True, exist_ok=True)

MLB_CHEST = Path(r"D:\Project\Radiology_RAG\data\MedLayBench-V\chest")
DATASETS = {
    "mlbv_chest_validation": MLB_CHEST/"validation_chest_text_final.parquet",   # tuning split
    "mlbv_chest_test":       MLB_CHEST/"test_chest_text_final.parquet",         # rules frozen before use
}
EXPERT_COL = "expert_caption"
LAY_COLS   = ("lay_caption", "layman_caption")       # the first one present in the file is used

# Same matching parameters as the normalized coverage run (coverage notebook Cells 9 and 13)
MAX_WORDS = 8
NORM_STOP = frozenset({"of", "to", "and", "with", "by", "in", "on", "the", "nos", "for", "s"})
WS, TOKEN = re.compile(r"\s+"), re.compile(r"[a-z0-9]+")

def norm(s):
    """Lowercase, hyphens -> spaces, collapse whitespace."""
    return WS.sub(" ", s.lower().replace("-", " ")).strip()

def key(s):
    """norm(), then alphanumeric tokens joined by spaces."""
    return " ".join(TOKEN.findall(norm(s)))

def bad_key(k):
    """Keys that are not concepts here: English stopwords and pure numbers."""
    return k in ENGLISH_STOP_WORDS or k.replace(" ", "").isdigit()

# Column positions for every RRF file, read from MRFILES
UMLS_COLS = {}
with open(UMLS_META_DIR/"MRFILES.RRF", encoding="utf-8") as fr:
    for line in fr:
        f = line.split("|")
        UMLS_COLS[f[0].removesuffix(".RRF")] = f[2].split(",")
COL = {t: {c: i for i, c in enumerate(cols)} for t, cols in UMLS_COLS.items()}

# NLM Semantic Groups (McCray et al., 2001), the copy saved by the coverage subtask
TUI_GROUP, TUI_NAME, GROUP_NAME = {}, {}, {}
with open(REF_OUT/"SemGroups.txt", encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            grp, grp_name, tui, sty = line.rstrip("\r\n").split("|")
            TUI_GROUP[tui], TUI_NAME[tui], GROUP_NAME[grp] = grp, sty, grp_name
print(f"{len(TUI_GROUP)} semantic types in {len(GROUP_NAME)} groups | MRXNS_ENG cols: {UMLS_COLS['MRXNS_ENG']}")

In [ ]:
# Cell G2 — load expert and lay captions, tokenized with the same key(). Peak RAM ~30 MB.
def toks_of(t):
    return key(t).split() if isinstance(t, str) else []

CAPS, TEXT, IDS = {}, {}, {}
for ds, path in DATASETS.items():
    names = pq.read_schema(path).names
    lay_col = next((c for c in LAY_COLS if c in names), None)
    assert lay_col, f"{path.name} has no lay caption column; columns are: {names}"
    id_col = "id" if "id" in names else None
    df = pd.read_parquet(path, columns=[EXPERT_COL, lay_col] + ([id_col] if id_col else []))
    CAPS[ds] = {"expert": [toks_of(t) for t in df[EXPERT_COL]],
                "lay":    [toks_of(t) for t in df[lay_col]]}
    TEXT[ds] = {"expert": df[EXPERT_COL].tolist(), "lay": df[lay_col].tolist()}
    IDS[ds]  = df[id_col].astype(str).tolist() if id_col else [str(i) for i in range(len(df))]
    del df
    print(f"{ds:<24} {len(CAPS[ds]['expert']):,} captions | lay column '{lay_col}' | id column {id_col}")

VOCAB = {t for d in CAPS.values() for side in d.values() for toks in side for t in toks}
print(f"distinct words (expert + lay): {len(VOCAB):,}")

In [ ]:
# Cell G3 — uninflection table for caption words. Two streaming passes (~535 MB + ~1.1 GB), ~2 min.
# Peak RAM: a few MB (dicts keyed by caption words only).
def plural_candidates(w):
    """English and Latin plural -> singular candidates. Fallback only; a candidate is accepted
    only if it occurs as a word in NLM's normalized strings."""
    if len(w) < 4:
        return []
    c = []
    if w.endswith("ies"):  c.append(w[:-3] + "y")                  # arteries -> artery
    if w.endswith("ices"): c += [w[:-4] + "ex", w[:-4] + "ix"]     # apices -> apex
    if w.endswith("ses"):  c.append(w[:-3] + "sis")                # metastases -> metastasis
    if w.endswith("es"):   c.append(w[:-2])                        # masses -> mass
    if w.endswith("s") and not w.endswith("ss"): c.append(w[:-1])  # consolidations -> consolidation
    if w.endswith("ae"):   c.append(w[:-1])                        # vertebrae -> vertebra
    if w.endswith("i"):    c.append(w[:-1] + "us")                 # bronchi -> bronchus
    return c

# Pass A: SUIs of single-word kept names that are also caption words
C = COL["MRCONSO"]
sui_word = {}
with open(REF_OUT/"MRCONSO.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        k = key(f[C["STR"]])
        if k in VOCAB:                         # VOCAB holds single words, so k is a single word
            sui_word[f[C["SUI"]]] = k

# Pass B: NLM's normalized form of those names, and which candidate words NLM's vocabulary contains
X = COL["MRXNS_ENG"]
CAND = VOCAB | {c for w in VOCAB for c in plural_candidates(w)}
base, in_vocab = {}, set()
t0 = time.time()
with open(UMLS_META_DIR/"MRXNS_ENG.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        nstr = f[X["NSTR"]]
        w = sui_word.get(f[X["SUI"]])
        if w is not None and " " not in nstr:
            base.setdefault(w, set()).add(nstr)
        for t in nstr.split():
            if t in CAND:
                in_vocab.add(t)
print(f"MRXNS_ENG pass: {time.time()-t0:.0f}s")

def uninflect(w):
    """(base form, source). Priority: NLM's own normal form of that exact name > word already
    in NLM's vocabulary > plural rule whose result is in NLM's vocabulary > unchanged."""
    b = base.get(w)
    if b:
        return (w, "unchanged") if w in b else (min(b), "umls_name")
    if w in in_vocab:
        return w, "unchanged"
    for c in plural_candidates(w):
        if c in in_vocab:
            return c, "plural_rule"
    return w, "unchanged"

LEMMA, src = {}, Counter()
with open(GOLD_OUT/"uninflection_map.csv", "w", encoding="utf-8", newline="") as fh:
    w_ = csv.writer(fh); w_.writerow(["caption_word", "base", "source"])
    for w in sorted(VOCAB):
        b, s = uninflect(w)
        LEMMA[w] = b
        src[s] += 1
        if s != "unchanged":
            w_.writerow([w, b, s])
del sui_word, base, CAND; gc.collect()
print("sources:", dict(src))

In [ ]:
# Cell G4 — normalized n-grams of expert AND lay captions -> lexicon from MRXNS_ENG, restricted to
# names kept in reference_data. In the same MRCONSO pass, also collect what Step 2 needs:
#   SRC[(CUI, key)]  = sources (SAB) that give this CUI a name with this normalized key
#   PREF             = (CUI, key) pairs where the key is the CUI's UMLS preferred name (TS=P, STT=PF, ISPREF=Y)
# Two streaming passes, ~2 min. Peak RAM: NGRAMS (~150 MB) while building, then tens of MB.
def nkey(toks):
    """NLM-style normalized key of a token span: drop NORM_STOP, uninflect, sort the words."""
    return " ".join(sorted(LEMMA.get(t, t) for t in toks if t not in NORM_STOP))

def spans(toks):
    """Spans of 1..MAX_WORDS tokens that start and end on a non-stopword token."""
    for i in range(len(toks)):
        if toks[i] in NORM_STOP:
            continue
        for n in range(1, min(MAX_WORDS, len(toks) - i) + 1):
            if toks[i + n - 1] not in NORM_STOP:
                yield i, n

NGRAMS = set()
for d in CAPS.values():
    for side in d.values():
        for toks in side:
            for i, n in spans(toks):
                NGRAMS.add(nkey(toks[i:i+n]))
print(f"distinct normalized n-grams (expert + lay): {len(NGRAMS):,}")

cand = {}                                      # (CUI, SUI) -> {normalized key}
with open(UMLS_META_DIR/"MRXNS_ENG.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        k = f[X["NSTR"]]
        if k in NGRAMS and not bad_key(k):
            cand.setdefault((f[X["CUI"]], f[X["SUI"]]), set()).add(k)
del NGRAMS; gc.collect()
cand_cuis = {c for c, _ in cand}

LEX, SRC, PREF = defaultdict(set), defaultdict(set), set()
PREF_NAME, ANY_NAME = {}, {}                   # display names for audit files
with open(REF_OUT/"MRCONSO.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        cui = f[C["CUI"]]
        if cui not in cand_cuis:
            continue
        is_pref = f[C["TS"]] == "P" and f[C["STT"]] == "PF" and f[C["ISPREF"]] == "Y"
        if is_pref:
            PREF_NAME[cui] = f[C["STR"]]
        ANY_NAME.setdefault(cui, f[C["STR"]])
        for k in cand.get((cui, f[C["SUI"]]), ()):
            LEX[k].add(cui)
            SRC[(cui, k)].add(f[C["SAB"]])
            if is_pref:
                PREF.add((cui, k))
del cand, cand_cuis; gc.collect()
LEX = dict(LEX)

def name(c):
    return PREF_NAME.get(c) or ANY_NAME.get(c, "")

amb = sum(len(v) > 1 for v in LEX.values())
print(f"lexicon keys {len(LEX):,} | ambiguous {amb:,} ({amb/len(LEX):.1%}) | "
      f"(CUI, key) pairs {len(SRC):,} | preferred-name pairs {len(PREF):,}")

In [ ]:
# Cell G5 — TUIs of matched CUIs only. Streams reference_data/MRSTY.RRF. Peak RAM: a few MB.
matched = {c for v in LEX.values() for c in v}
S = COL["MRSTY"]
CUI_TUIS = defaultdict(set)
with open(REF_OUT/"MRSTY.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        cui = line[:line.index("|")]
        if cui in matched:
            CUI_TUIS[cui].add(line.split("|")[S["TUI"]])
missing = matched - CUI_TUIS.keys()
assert not missing, f"{len(missing)} matched CUIs have no semantic type, e.g. {sorted(missing)[:5]}"
print(f"matched concepts {len(matched):,}, all with semantic types")

In [ ]:
# Cell G6 — STEP 0: greedy longest-match extraction on EXPERT captions. Peak RAM: a few MB.
# Longest match means nested matches are never emitted ("effusion" inside "pleural effusion").
def extract(toks):
    """Left to right; at each position take the longest span whose normalized key is in LEX."""
    i, out = 0, []
    while i < len(toks):
        hit = None
        if toks[i] not in NORM_STOP:
            for n in range(min(MAX_WORDS, len(toks) - i), 0, -1):
                if toks[i + n - 1] in NORM_STOP:
                    continue
                k = nkey(toks[i:i+n])
                if k in LEX:
                    hit = (i, n, k)
                    break
        if hit:
            out.append(hit)
            i += hit[1]
        else:
            i += 1
    return out

COVERAGE_RUN = {"mlbv_chest_validation": 30_861, "mlbv_chest_test": 30_389}   # coverage notebook Cell 18
MENTIONS = {}                                  # ds -> [(row, start, n_words, span, key, cuis)]
for ds, d in CAPS.items():
    MENTIONS[ds] = [(row, s, n, " ".join(toks[s:s+n]), k, tuple(sorted(LEX[k])))
                    for row, toks in enumerate(d["expert"]) for s, n, k in extract(toks)]
    amb = sum(len(m[5]) > 1 for m in MENTIONS[ds])
    print(f"{ds:<24} {len(MENTIONS[ds]):,} mentions (coverage run: {COVERAGE_RUN[ds]:,}, should match) | "
          f"ambiguous {amb:,} ({amb/len(MENTIONS[ds]):.1%})")

In [ ]:
# Cell G7 — STEP 1: which semantic types can be gold. Peak RAM ~0.
# Base: the coverage subtask's CLINICAL_GROUPS minus LIVB, so "gold" stays a subset of "clinical".
GOLD_GROUPS = {
    "DISO": "Disorders: findings and diseases, the main thing a lay reader needs explained",
    "ANAT": "Anatomy: where a finding is",
    "PROC": "Procedures: the imaging study and interventions named in the caption",
    "DEVI": "Devices: tubes, lines and implants visible on the image",
    "CHEM": "Chemicals & Drugs: contrast agents and substances named in the caption",
    "PHYS": "Physiology: body functions and clinical attributes",
    "PHEN": "Phenomena: e.g. test results and biologic processes",
}
# Not gold: CONC (generic ideas: 'left', 'image', 'size'), ACTI, OBJC, OCCU, ORGA, and LIVB, whose
# remaining types after the reference filter are people and groups ('patient', 'man', 'infant').

# Exceptions kept inside a non-gold group
KEEP_TYPES = {
    "T004": "pathogen named as the cause of a finding",   # Fungus (LIVB)
    "T005": "pathogen named as the cause of a finding",   # Virus (LIVB)
    "T007": "pathogen named as the cause of a finding",   # Bacterium (LIVB)
}
# Single types dropped inside gold groups: molecular or research level, never an imaging concept
EXCLUDE_TYPES = {
    "T044": "molecular level",                            # Molecular Function (PHYS)
    "T045": "molecular level",                            # Genetic Function (PHYS)
    "T062": "research, not care",                         # Research Activity (PROC)
    "T063": "molecular research technique",               # Molecular Biology Research Technique (PROC)
}
# Step 2 tie-break order: captions report findings (DISO) located in anatomy (ANAT), then what was done
GROUP_PRIORITY = ["DISO", "ANAT", "PROC", "DEVI", "CHEM", "PHYS", "PHEN", "LIVB"]

# Guards: the policy must be consistent with SemGroups
for t in {**KEEP_TYPES, **EXCLUDE_TYPES}:
    assert t in TUI_GROUP, f"{t} is not in SemGroups"
for t in KEEP_TYPES:
    assert TUI_GROUP[t] not in GOLD_GROUPS, f"{t} is an exception but its group is already gold"
for t in EXCLUDE_TYPES:
    assert TUI_GROUP[t] in GOLD_GROUPS, f"{t} is excluded but its group is not gold anyway"
assert set(GROUP_PRIORITY) >= set(GOLD_GROUPS) | {TUI_GROUP[t] for t in KEEP_TYPES}

def keep_tui(t):
    return t not in EXCLUDE_TYPES and (TUI_GROUP[t] in GOLD_GROUPS or t in KEEP_TYPES)

def keep_cui(c):
    """A CUI can be gold if at least one of its types is allowed."""
    return any(keep_tui(t) for t in CUI_TUIS[c])

for t in sorted({*KEEP_TYPES, *EXCLUDE_TYPES}):
    print(f"  {'keep' if t in KEEP_TYPES else 'drop'}  {t}  {TUI_GROUP[t]}  {TUI_NAME[t]}")

In [ ]:
# Cell G8 — STEP 2: one CUI per mention, with the deciding rule logged. Peak RAM ~15 MB (mention dicts).
G_RANK = {g: i for i, g in enumerate(GROUP_PRIORITY)}

def rank(c):
    return min(G_RANK[TUI_GROUP[t]] for t in CUI_TUIS[c] if keep_tui(t))

def choose(k, cands):
    """Cascade: unique > group priority > UMLS preferred name > most sources > lowest CUI."""
    if len(cands) == 1:
        return cands[0], "unique"
    best = min(rank(c) for c in cands)
    s = [c for c in cands if rank(c) == best]
    if len(s) == 1:
        return s[0], "group_priority"
    p = [c for c in s if (c, k) in PREF]
    if len(p) == 1:
        return p[0], "preferred_name"
    s = p or s
    top = max(len(SRC.get((c, k), ())) for c in s)
    s = [c for c in s if len(SRC.get((c, k), ())) == top]
    if len(s) == 1:
        return s[0], "source_count"
    return min(s), "lowest_cui"                # reproducibility only; its share should be small

DECIDED = {}
for ds, ments in MENTIONS.items():
    out, rules = [], Counter()
    for row, s, n, span, k, cuis in ments:
        kept = [c for c in cuis if keep_cui(c)]
        if kept:
            c, rule = choose(k, kept)
        else:
            c, rule = "", "dropped_by_group"
        rules[rule] += 1
        out.append({"row": row, "start": s, "n_words": n, "span": span, "key": k,
                    "cuis_all": cuis, "cuis_kept": kept, "chosen": c, "rule": rule})
    DECIDED[ds] = out
    print(f"{ds}: " + ", ".join(f"{r} {v:,} ({v/len(ments):.1%})" for r, v in rules.most_common()))

# Audit of ambiguous keys on the VALIDATION split (the tuning split), most frequent first
DS = "mlbv_chest_validation"
freq = Counter(m["key"] for m in DECIDED[DS] if len(m["cuis_kept"]) > 1)
first = {}
for m in DECIDED[DS]:
    first.setdefault(m["key"], m)
with open(GOLD_OUT/f"{DS}_disambiguation_audit.csv", "w", encoding="utf-8-sig", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["key", "mentions", "n_candidates", "chosen", "chosen_name", "rule", "candidates"])
    for k, v in freq.most_common():
        m = first[k]
        cands = " || ".join(
            f"{c}: {name(c)} [{','.join(sorted({TUI_GROUP[t] for t in CUI_TUIS[c]}))}; "
            f"{len(SRC.get((c, k), ()))} src{'; PREF' if (c, k) in PREF else ''}]"
            for c in m["cuis_kept"])
        w.writerow([k, v, len(m["cuis_kept"]), m["chosen"], name(m["chosen"]), m["rule"], cands])
print(f"audit: {len(freq):,} ambiguous keys -> {DS}_disambiguation_audit.csv")

In [ ]:
# Cell G9 — STEP 3: drop a term if the lay caption keeps the same normalized term, then build G.
# The comparison is by STRING, not by CUI: 'collapsed lung' is a UMLS name for pneumothorax, so a CUI
# comparison would drop exactly the terms that WERE simplified.
#   gold_all  = CUIs after Steps 0-2
#   gold_hard = gold_all minus CUIs whose every mention also appears verbatim (normalized) in the lay caption
# Peak RAM: a few MB.
def lay_span_keys(toks):
    return {nkey(toks[i:i+n]) for i, n in spans(toks)}

SUMMARY, GOLD = [], {}
for ds, dec in DECIDED.items():
    lay = CAPS[ds]["lay"]
    cur_row, cur_keys = None, set()
    for m in dec:                                    # mentions are in row order
        if m["row"] != cur_row:
            cur_row, cur_keys = m["row"], lay_span_keys(lay[m["row"]])
        m["in_lay"] = bool(m["chosen"]) and m["key"] in cur_keys

    with open(GOLD_OUT/f"{ds}_mentions.tsv", "w", encoding="utf-8", newline="") as fh:
        fh.write("row\tstart\tn_words\tspan\tkey\tcuis_all\tcuis_kept\tchosen\trule\tin_lay\n")
        for m in dec:
            fh.write(f"{m['row']}\t{m['start']}\t{m['n_words']}\t{m['span']}\t{m['key']}\t"
                     f"{';'.join(m['cuis_all'])}\t{';'.join(m['cuis_kept'])}\t{m['chosen']}\t"
                     f"{m['rule']}\t{m['in_lay']}\n")

    by_row = defaultdict(list)
    for m in dec:
        if m["chosen"]:
            by_row[m["row"]].append(m)
    GOLD[ds] = {}
    with open(GOLD_OUT/f"{ds}_gold.tsv", "w", encoding="utf-8", newline="") as fh:
        fh.write("row\tid\tn_mentions\tgold_all\tgold_hard\tgold_hard_names\n")
        for row in range(len(CAPS[ds]["expert"])):
            ms = by_row.get(row, [])
            g_all  = sorted({m["chosen"] for m in ms})
            g_hard = sorted({m["chosen"] for m in ms if not m["in_lay"]})
            GOLD[ds][row] = (g_all, g_hard)
            fh.write(f"{row}\t{IDS[ds][row]}\t{len(ms)}\t{';'.join(g_all)}\t{';'.join(g_hard)}\t"
                     f"{' | '.join(name(c) for c in g_hard)}\n")

    n = len(GOLD[ds])
    sizes_all  = [len(a) for a, _ in GOLD[ds].values()]
    sizes_hard = [len(h) for _, h in GOLD[ds].values()]
    amb = [m for m in dec if m["chosen"] and m["rule"] != "unique"]
    SUMMARY.append({
        "dataset": ds, "captions": n,
        "mean_G_all": sum(sizes_all) / n, "mean_G_hard": sum(sizes_hard) / n,
        "captions_G_hard_empty": sum(s == 0 for s in sizes_hard),
        "distinct_cuis_all":  len({c for a, _ in GOLD[ds].values() for c in a}),
        "distinct_cuis_hard": len({c for _, h in GOLD[ds].values() for c in h}),
        "mentions_dropped_by_group": sum(m["rule"] == "dropped_by_group" for m in dec),
        "mentions_in_lay": sum(m["in_lay"] for m in dec),
        **{f"share_{r}": sum(m["rule"] == r for m in amb) / max(len(amb), 1)
           for r in ("group_priority", "preferred_name", "source_count", "lowest_cui")},
    })

summary = pd.DataFrame(SUMMARY)
summary.to_csv(GOLD_OUT/"gold_summary.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
summary.T

In [ ]:
# Cell G10 — STEP 4a: 50 random VALIDATION captions for manual checking. Peak RAM ~0.
# Open the CSV in Excel and fill 'verdict' on every line:
#   ok        = right concept AND right CUI
#   wrong_cui = right concept, wrong CUI (write the right one in 'correct_cui')
#   not_gold  = should not be a gold concept at all
# For every concept the extractor MISSED, add a new line with row, span and verdict = missed.
# This checks Steps 0-2 (gold_all). Step 3 is a fixed string rule and needs no manual check.
DS, N_CHECK, SEED = "mlbv_chest_validation", 50, 42
rows = sorted(random.Random(SEED).sample(range(len(CAPS[DS]["expert"])), N_CHECK))
by_row = defaultdict(list)
for m in DECIDED[DS]:
    if m["chosen"]:
        by_row[m["row"]].append(m)

with open(GOLD_OUT/"manual_check_50.csv", "w", encoding="utf-8-sig", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["row", "id", "expert_caption", "lay_caption", "span", "cui", "name", "groups",
                "rule", "n_candidates", "in_lay", "verdict", "correct_cui", "note"])
    for row in rows:
        for j, m in enumerate(by_row.get(row, [])):
            c = m["chosen"]
            w.writerow([row, IDS[DS][row],
                        TEXT[DS]["expert"][row] if j == 0 else "",
                        TEXT[DS]["lay"][row] if j == 0 else "",
                        m["span"], c, name(c),
                        ",".join(sorted({TUI_GROUP[t] for t in CUI_TUIS[c]})),
                        m["rule"], len(m["cuis_kept"]), m["in_lay"], "", "", ""])
print(f"{sum(len(by_row.get(r, [])) for r in rows)} lines for {N_CHECK} captions -> manual_check_50.csv")

In [ ]:
# Cell G11 — score the filled manual check. Mention-level. Peak RAM ~0.
#   TP = ok | FP = wrong_cui + not_gold | FN = wrong_cui + missed  (a wrong CUI also misses the right one)
# Only captions whose lines ALL have a verdict are scored, so a partly filled file works.
VALID = {"ok", "wrong_cui", "not_gold", "missed"}
ann = pd.read_csv(GOLD_OUT/"manual_check_50.csv", dtype=str, encoding="utf-8-sig").fillna("")
ann["v"] = ann["verdict"].str.strip().str.lower()

typos = sorted(set(ann["v"]) - VALID - {""})
assert not typos, f"unknown verdicts (check spelling): {typos}"

done_rows = ann.groupby("row")["v"].apply(lambda s: (s != "").all())
done = ann[ann["row"].isin(done_rows[done_rows].index)]
print(f"captions fully annotated: {done_rows.sum()}/{len(done_rows)} | "
      f"lines annotated: {(ann['v'] != '').sum()}/{len(ann)}")
assert len(done), "no caption is fully annotated yet: fill every line of at least one caption"

cnt = done["v"].value_counts()
tp, wrong = cnt.get("ok", 0), cnt.get("wrong_cui", 0)
fp, fn = wrong + cnt.get("not_gold", 0), wrong + cnt.get("missed", 0)
P = tp / (tp + fp) if tp + fp else float("nan")
R = tp / (tp + fn) if tp + fn else float("nan")
F1 = 2*P*R/(P+R) if P + R else float("nan")
print(f"precision {P:.3f} | recall {R:.3f} | F1 {F1:.3f} | counts {dict(cnt)}")

# Disambiguation accuracy per rule: among lines whose concept was right, was the CUI right?
dis = done[done["v"].isin(["ok", "wrong_cui"])]
print(dis.groupby("rule")["v"].agg(lines="size", accuracy=lambda s: (s == "ok").mean()))

# Which rule produced the not_gold lines (tells you what to fix next)
print(done[done["v"] == "not_gold"].groupby("rule").size().rename("not_gold_lines"))

# Attempt 02

In [ ]:
# Cell G1 — setup. Peak RAM ~0.
# Gold-concept subtask (separate from KB preprocessing). Read-only inputs:
#   META/MRFILES.RRF, META/MRXNS_ENG.RRF                     (UMLS 2026AA)
#   reference_data/MRCONSO.RRF, MRSTY.RRF, SemGroups.txt     (general clinical English UMLS, NOT the chest KB)
#   MedLayBench-V chest validation/test parquet              (expert + lay captions)
# Gold concepts never look at the KB, so retrieval recall and the KB ceiling are not circular.
import csv, gc, random, re, time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS   # scikit-learn's 318-word list

UMLS_META_DIR = Path(r"D:\Project\Radiology_RAG\data\raw_data\umls\umls_2026AA_full\2026AA\META")
REF_OUT  = Path(r"D:\Project\Radiology_RAG\data\subtask_data_coverage\reference_data")
GOLD_OUT = Path(r"D:\Project\Radiology_RAG\data\subtask_gold_concepts_det")
GOLD_OUT.mkdir(parents=True, exist_ok=True)

MLB_CHEST = Path(r"D:\Project\Radiology_RAG\data\MedLayBench-V\chest")
DATASETS = {
    "mlbv_chest_validation": MLB_CHEST/"validation_chest_text_final.parquet",   # tuning split
    "mlbv_chest_test":       MLB_CHEST/"test_chest_text_final.parquet",         # rules frozen before use
}
EXPERT_COL = "expert_caption"
LAY_COLS   = ("lay_caption", "layman_caption")       # the first one present in the file is used

# Same matching parameters as the normalized coverage run (coverage notebook Cells 9 and 13)
MAX_WORDS = 8
NORM_STOP = frozenset({"of", "to", "and", "with", "by", "in", "on", "the", "nos", "for", "s"})
WS, TOKEN = re.compile(r"\s+"), re.compile(r"[a-z0-9]+")

def norm(s):
    """Lowercase, hyphens -> spaces, collapse whitespace."""
    return WS.sub(" ", s.lower().replace("-", " ")).strip()

def key(s):
    """norm(), then alphanumeric tokens joined by spaces."""
    return " ".join(TOKEN.findall(norm(s)))

def bad_key(k):
    """Keys that are not concepts here: English stopwords and pure numbers."""
    return k in ENGLISH_STOP_WORDS or k.replace(" ", "").isdigit()

# Column positions for every RRF file, read from MRFILES
UMLS_COLS = {}
with open(UMLS_META_DIR/"MRFILES.RRF", encoding="utf-8") as fr:
    for line in fr:
        f = line.split("|")
        UMLS_COLS[f[0].removesuffix(".RRF")] = f[2].split(",")
COL = {t: {c: i for i, c in enumerate(cols)} for t, cols in UMLS_COLS.items()}

# NLM Semantic Groups (McCray et al., 2001), the copy saved by the coverage subtask
TUI_GROUP, TUI_NAME, GROUP_NAME = {}, {}, {}
with open(REF_OUT/"SemGroups.txt", encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            grp, grp_name, tui, sty = line.rstrip("\r\n").split("|")
            TUI_GROUP[tui], TUI_NAME[tui], GROUP_NAME[grp] = grp, sty, grp_name
print(f"{len(TUI_GROUP)} semantic types in {len(GROUP_NAME)} groups | MRXNS_ENG cols: {UMLS_COLS['MRXNS_ENG']}")

In [ ]:
# Cell G2 — load expert and lay captions, tokenized with the same key(). Peak RAM ~30 MB.
# Sentence boundaries become a BOUNDARY token so no span can run into the next sentence
# ("CECT of the thorax. The image" must not match 'thorax image' -> Chest imaging).
BOUNDARY = "<s>"                                   # never occurs in a UMLS key
SENT_END = re.compile(r"[.;:!?](?=\s|[A-Z]|$)")    # '1.2-cm' is not split; 'discharge.Chest' is

def toks_of(t):
    if not isinstance(t, str):
        return []
    out = []
    for sent in SENT_END.split(t):
        toks = key(sent).split()
        if toks:
            if out:
                out.append(BOUNDARY)
            out += toks
    return out

CAPS, TEXT, IDS = {}, {}, {}
for ds, path in DATASETS.items():
    names = pq.read_schema(path).names
    lay_col = next((c for c in LAY_COLS if c in names), None)
    assert lay_col, f"{path.name} has no lay caption column; columns are: {names}"
    id_col = "id" if "id" in names else None
    df = pd.read_parquet(path, columns=[EXPERT_COL, lay_col] + ([id_col] if id_col else []))
    CAPS[ds] = {"expert": [toks_of(t) for t in df[EXPERT_COL]],
                "lay":    [toks_of(t) for t in df[lay_col]]}
    TEXT[ds] = {"expert": df[EXPERT_COL].tolist(), "lay": df[lay_col].tolist()}
    IDS[ds]  = df[id_col].astype(str).tolist() if id_col else [str(i) for i in range(len(df))]
    del df
    print(f"{ds:<24} {len(CAPS[ds]['expert']):,} captions | lay column '{lay_col}' | id column {id_col}")

VOCAB = {t for d in CAPS.values() for side in d.values() for toks in side for t in toks if t != BOUNDARY}
print(f"distinct words (expert + lay): {len(VOCAB):,}")

In [ ]:
# Cell G3 — uninflection table for caption words. Two streaming passes (~535 MB + ~1.1 GB), ~2 min.
# Peak RAM: a few MB (dicts keyed by caption words only).
def plural_candidates(w):
    """English and Latin plural -> singular candidates. Fallback only; a candidate is accepted
    only if it occurs as a word in NLM's normalized strings."""
    if len(w) < 4:
        return []
    c = []
    if w.endswith("ies"):  c.append(w[:-3] + "y")                  # arteries -> artery
    if w.endswith("ices"): c += [w[:-4] + "ex", w[:-4] + "ix"]     # apices -> apex
    if w.endswith("ses"):  c.append(w[:-3] + "sis")                # metastases -> metastasis
    if w.endswith("es"):   c.append(w[:-2])                        # masses -> mass
    if w.endswith("s") and not w.endswith("ss"): c.append(w[:-1])  # consolidations -> consolidation
    if w.endswith("ae"):   c.append(w[:-1])                        # vertebrae -> vertebra
    if w.endswith("i"):    c.append(w[:-1] + "us")                 # bronchi -> bronchus
    return c

# Pass A: SUIs of single-word kept names that are also caption words
C = COL["MRCONSO"]
sui_word = {}
with open(REF_OUT/"MRCONSO.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        k = key(f[C["STR"]])
        if k in VOCAB:                         # VOCAB holds single words, so k is a single word
            sui_word[f[C["SUI"]]] = k

# Pass B: NLM's normalized form of those names, and which candidate words NLM's vocabulary contains
X = COL["MRXNS_ENG"]
CAND = VOCAB | {c for w in VOCAB for c in plural_candidates(w)}
base, in_vocab = {}, set()
t0 = time.time()
with open(UMLS_META_DIR/"MRXNS_ENG.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        nstr = f[X["NSTR"]]
        w = sui_word.get(f[X["SUI"]])
        if w is not None and " " not in nstr:
            base.setdefault(w, set()).add(nstr)
        for t in nstr.split():
            if t in CAND:
                in_vocab.add(t)
print(f"MRXNS_ENG pass: {time.time()-t0:.0f}s")

def uninflect(w):
    """(base form, source). Priority: NLM's own normal form of that exact name > word already
    in NLM's vocabulary > plural rule whose result is in NLM's vocabulary > unchanged."""
    b = base.get(w)
    if b:
        return (w, "unchanged") if w in b else (min(b), "umls_name")
    if w in in_vocab:
        return w, "unchanged"
    for c in plural_candidates(w):
        if c in in_vocab:
            return c, "plural_rule"
    return w, "unchanged"

LEMMA, src = {}, Counter()
with open(GOLD_OUT/"uninflection_map.csv", "w", encoding="utf-8", newline="") as fh:
    w_ = csv.writer(fh); w_.writerow(["caption_word", "base", "source"])
    for w in sorted(VOCAB):
        b, s = uninflect(w)
        LEMMA[w] = b
        src[s] += 1
        if s != "unchanged":
            w_.writerow([w, b, s])
del sui_word, base, CAND; gc.collect()
print("sources:", dict(src))

In [ ]:
# Cell G4 — normalized n-grams of expert AND lay captions -> lexicon from MRXNS_ENG, restricted to
# names kept in reference_data. In the same MRCONSO pass, also collect what Step 2 needs:
#   SRC[(CUI, key)]  = sources (SAB) that give this CUI a name with this normalized key
#   PREF             = (CUI, key) pairs where the key is the CUI's UMLS preferred name (TS=P, STT=PF, ISPREF=Y)
# Two streaming passes, ~2 min. Peak RAM: NGRAMS (~150 MB) while building, then tens of MB.

# Span rules (apply to expert extraction AND the lay-caption check in Step 3):
#  - a span never crosses a sentence boundary or a coordinator: normalization sorts words, so
#    'superior vena cava and the right [atrium]' would otherwise become 'Right superior vena cava'
#  - a span never starts or ends with an NLM stopword or a determiner ('a CT' -> an enzyme name 'CT A')
EDGE_STOP = frozenset({"a", "an", "this", "that", "these", "those", "its", "their", "his", "her"})
BAD_EDGE  = NORM_STOP | EDGE_STOP | {BOUNDARY}
WALL      = frozenset({BOUNDARY, "and", "or"})

def nkey(toks):
    """NLM-style normalized key of a token span: drop NORM_STOP, uninflect, sort the words."""
    return " ".join(sorted(LEMMA.get(t, t) for t in toks if t not in NORM_STOP))

def span_lengths(toks, i):
    """Valid span lengths (ascending) for a span starting at position i."""
    if toks[i] in BAD_EDGE:
        return
    for n in range(1, min(MAX_WORDS, len(toks) - i) + 1):
        t = toks[i + n - 1]
        if t in WALL:
            return
        if t not in BAD_EDGE:
            yield n

def spans(toks):
    for i in range(len(toks)):
        for n in span_lengths(toks, i):
            yield i, n

NGRAMS = set()
for d in CAPS.values():
    for side in d.values():
        for toks in side:
            for i, n in spans(toks):
                NGRAMS.add(nkey(toks[i:i+n]))
print(f"distinct normalized n-grams (expert + lay): {len(NGRAMS):,}")

cand = {}                                      # (CUI, SUI) -> {normalized key}
with open(UMLS_META_DIR/"MRXNS_ENG.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        k = f[X["NSTR"]]
        if k in NGRAMS and not bad_key(k):
            cand.setdefault((f[X["CUI"]], f[X["SUI"]]), set()).add(k)
del NGRAMS; gc.collect()
cand_cuis = {c for c, _ in cand}

LEX, SRC, PREF = defaultdict(set), defaultdict(set), set()
PREF_NAME, ANY_NAME = {}, {}                   # display names for audit files
with open(REF_OUT/"MRCONSO.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.split("|")
        cui = f[C["CUI"]]
        if cui not in cand_cuis:
            continue
        is_pref = f[C["TS"]] == "P" and f[C["STT"]] == "PF" and f[C["ISPREF"]] == "Y"
        if is_pref:
            PREF_NAME[cui] = f[C["STR"]]
        ANY_NAME.setdefault(cui, f[C["STR"]])
        for k in cand.get((cui, f[C["SUI"]]), ()):
            LEX[k].add(cui)
            SRC[(cui, k)].add(f[C["SAB"]])
            if is_pref:
                PREF.add((cui, k))
del cand, cand_cuis; gc.collect()
LEX = dict(LEX)

def name(c):
    return PREF_NAME.get(c) or ANY_NAME.get(c, "")

amb = sum(len(v) > 1 for v in LEX.values())
print(f"lexicon keys {len(LEX):,} | ambiguous {amb:,} ({amb/len(LEX):.1%}) | "
      f"(CUI, key) pairs {len(SRC):,} | preferred-name pairs {len(PREF):,}")

In [ ]:
# Cell G5 — TUIs of matched CUIs only. Streams reference_data/MRSTY.RRF. Peak RAM: a few MB.
matched = {c for v in LEX.values() for c in v}
S = COL["MRSTY"]
CUI_TUIS = defaultdict(set)
with open(REF_OUT/"MRSTY.RRF", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        cui = line[:line.index("|")]
        if cui in matched:
            CUI_TUIS[cui].add(line.split("|")[S["TUI"]])
missing = matched - CUI_TUIS.keys()
assert not missing, f"{len(missing)} matched CUIs have no semantic type, e.g. {sorted(missing)[:5]}"
print(f"matched concepts {len(matched):,}, all with semantic types")

In [ ]:
# Cell G6 — STEP 0: greedy longest-match extraction on EXPERT captions. Peak RAM: a few MB.
# Longest match means nested matches are never emitted ("effusion" inside "pleural effusion").
# Counts are now LOWER than the coverage run (30,861 / 30,389) by design: the new span rules
# remove cross-sentence, article-initial and cross-'and' matches.
def extract(toks):
    """Left to right; at each position take the longest valid span whose normalized key is in LEX."""
    i, out = 0, []
    while i < len(toks):
        hit = None
        for n in sorted(span_lengths(toks, i), reverse=True):
            k = nkey(toks[i:i+n])
            if k in LEX:
                hit = (i, n, k)
                break
        if hit:
            out.append(hit)
            i += hit[1]
        else:
            i += 1
    return out

MENTIONS = {}                                  # ds -> [(row, start, n_words, span, key, cuis)]
for ds, d in CAPS.items():
    MENTIONS[ds] = [(row, s, n, " ".join(toks[s:s+n]), k, tuple(sorted(LEX[k])))
                    for row, toks in enumerate(d["expert"]) for s, n, k in extract(toks)]
    amb = sum(len(m[5]) > 1 for m in MENTIONS[ds])
    print(f"{ds:<24} {len(MENTIONS[ds]):,} mentions | ambiguous {amb:,} ({amb/len(MENTIONS[ds]):.1%})")

In [ ]:
# Cell G7 — STEP 1: which semantic types can be gold. Peak RAM ~0.
# Base: the coverage subtask's CLINICAL_GROUPS minus LIVB, so "gold" stays a subset of "clinical".
GOLD_GROUPS = {
    "DISO": "Disorders: findings and diseases, the main thing a lay reader needs explained",
    "ANAT": "Anatomy: where a finding is",
    "PROC": "Procedures: the imaging study and interventions named in the caption",
    "DEVI": "Devices: tubes, lines and implants visible on the image",
    "CHEM": "Chemicals & Drugs: contrast agents and substances named in the caption",
    "PHYS": "Physiology: body functions and clinical attributes",
    "PHEN": "Phenomena: e.g. test results and biologic processes",
}
# Not gold: CONC (generic ideas: 'left', 'image', 'size'), ACTI, OBJC, OCCU, ORGA, and LIVB, whose
# remaining types after the reference filter are people and groups ('patient', 'man', 'infant').

# Exceptions kept inside a non-gold group
KEEP_TYPES = {
    "T004": "pathogen named as the cause of a finding",   # Fungus (LIVB)
    "T005": "pathogen named as the cause of a finding",   # Virus (LIVB)
    "T007": "pathogen named as the cause of a finding",   # Bacterium (LIVB)
}
# Single types dropped inside gold groups: molecular or research level, never an imaging concept
EXCLUDE_TYPES = {
    "T044": "molecular level",                            # Molecular Function (PHYS)
    "T045": "molecular level",                            # Genetic Function (PHYS)
    "T062": "research, not care",                         # Research Activity (PROC)
    "T063": "molecular research technique",               # Molecular Biology Research Technique (PROC)
}
# Step 2 tie-break order: captions report findings (DISO) located in anatomy (ANAT), then what was done
GROUP_PRIORITY = ["DISO", "ANAT", "PROC", "DEVI", "CHEM", "PHYS", "PHEN", "LIVB"]

# Guards: the policy must be consistent with SemGroups
for t in {**KEEP_TYPES, **EXCLUDE_TYPES}:
    assert t in TUI_GROUP, f"{t} is not in SemGroups"
for t in KEEP_TYPES:
    assert TUI_GROUP[t] not in GOLD_GROUPS, f"{t} is an exception but its group is already gold"
for t in EXCLUDE_TYPES:
    assert TUI_GROUP[t] in GOLD_GROUPS, f"{t} is excluded but its group is not gold anyway"
assert set(GROUP_PRIORITY) >= set(GOLD_GROUPS) | {TUI_GROUP[t] for t in KEEP_TYPES}

def keep_tui(t):
    return t not in EXCLUDE_TYPES and (TUI_GROUP[t] in GOLD_GROUPS or t in KEEP_TYPES)

def keep_cui(c):
    """A CUI can be gold if at least one of its types is allowed."""
    return any(keep_tui(t) for t in CUI_TUIS[c])

for t in sorted({*KEEP_TYPES, *EXCLUDE_TYPES}):
    print(f"  {'keep' if t in KEEP_TYPES else 'drop'}  {t}  {TUI_GROUP[t]}  {TUI_NAME[t]}")

In [ ]:
# Cell G8 — STEP 2 then STEP 1: decide the sense among ALL candidates, then apply the group filter
# to the winner. Filtering first removed the everyday sense ('left' the side, 'changes' = Change)
# and let a junk concept win ('left:' DISO, 'Menopause'). Peak RAM ~15 MB (mention dicts).
G_RANK = {g: i for i, g in enumerate(GROUP_PRIORITY)}
NOT_GOLD = len(G_RANK)

def rank(c):
    return min((G_RANK[TUI_GROUP[t]] if keep_tui(t) else NOT_GOLD) for t in CUI_TUIS[c])

def choose(k, cands):
    """Cascade over all candidates:
    unique > UMLS preferred name > most sources (most-frequent-sense proxy) > group priority > lowest CUI."""
    if len(cands) == 1:
        return cands[0], "unique"
    p = [c for c in cands if (c, k) in PREF]
    if len(p) == 1:
        return p[0], "preferred_name"
    s = p or list(cands)
    top = max(len(SRC.get((c, k), ())) for c in s)
    s = [c for c in s if len(SRC.get((c, k), ())) == top]
    if len(s) == 1:
        return s[0], "source_count"
    best = min(rank(c) for c in s)
    s = [c for c in s if rank(c) == best]
    if len(s) == 1:
        return s[0], "group_priority"
    return min(s), "lowest_cui"                # reproducibility only; its share should be small

DECIDED = {}
for ds, ments in MENTIONS.items():
    out, rules = [], Counter()
    for row, s, n, span, k, cuis in ments:
        winner, rule = choose(k, cuis)
        gold = keep_cui(winner)
        out.append({"row": row, "start": s, "n_words": n, "span": span, "key": k,
                    "cuis_all": cuis, "cuis_kept": [c for c in cuis if keep_cui(c)],
                    "winner": winner, "winner_rule": rule,
                    "chosen": winner if gold else "",
                    "rule": rule if gold else "dropped_by_group"})
        rules[out[-1]["rule"]] += 1
    DECIDED[ds] = out
    print(f"{ds}: " + ", ".join(f"{r} {v:,} ({v/len(ments):.1%})" for r, v in rules.most_common()))

# Audit of ambiguous keys on the VALIDATION split, most frequent first; '*' marks gold-eligible candidates
DS = "mlbv_chest_validation"
freq = Counter(m["key"] for m in DECIDED[DS] if len(m["cuis_all"]) > 1)
first = {}
for m in DECIDED[DS]:
    first.setdefault(m["key"], m)
with open(GOLD_OUT/f"{DS}_disambiguation_audit.csv", "w", encoding="utf-8-sig", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["key", "mentions", "n_candidates", "winner", "winner_name", "winner_rule",
                "gold_kept", "candidates"])
    for k, v in freq.most_common():
        m = first[k]
        cands = " || ".join(
            f"{'*' if keep_cui(c) else ''}{c}: {name(c)} "
            f"[{','.join(sorted({TUI_GROUP[t] for t in CUI_TUIS[c]}))}; "
            f"{len(SRC.get((c, k), ()))} src{'; PREF' if (c, k) in PREF else ''}]"
            for c in m["cuis_all"])
        w.writerow([k, v, len(m["cuis_all"]), m["winner"], name(m["winner"]), m["winner_rule"],
                    bool(m["chosen"]), cands])
print(f"audit: {len(freq):,} ambiguous keys -> {DS}_disambiguation_audit.csv")

In [ ]:
# Cell G9 — STEP 3: drop a term if the lay caption keeps the same normalized term, then build G.
# The comparison is by STRING, not by CUI: 'collapsed lung' is a UMLS name for pneumothorax, so a CUI
# comparison would drop exactly the terms that WERE simplified.
#   gold_all  = CUIs after Steps 0-2
#   gold_hard = gold_all minus CUIs whose every mention also appears verbatim (normalized) in the lay caption
# Peak RAM: a few MB.
def lay_span_keys(toks):
    return {nkey(toks[i:i+n]) for i, n in spans(toks)}

SUMMARY, GOLD = [], {}
for ds, dec in DECIDED.items():
    lay = CAPS[ds]["lay"]
    cur_row, cur_keys = None, set()
    for m in dec:                                    # mentions are in row order
        if m["row"] != cur_row:
            cur_row, cur_keys = m["row"], lay_span_keys(lay[m["row"]])
        m["in_lay"] = bool(m["chosen"]) and m["key"] in cur_keys

    with open(GOLD_OUT/f"{ds}_mentions.tsv", "w", encoding="utf-8", newline="") as fh:
        fh.write("row\tstart\tn_words\tspan\tkey\tcuis_all\tcuis_kept\tchosen\trule\tin_lay\n")
        for m in dec:
            fh.write(f"{m['row']}\t{m['start']}\t{m['n_words']}\t{m['span']}\t{m['key']}\t"
                     f"{';'.join(m['cuis_all'])}\t{';'.join(m['cuis_kept'])}\t{m['chosen']}\t"
                     f"{m['rule']}\t{m['in_lay']}\n")

    by_row = defaultdict(list)
    for m in dec:
        if m["chosen"]:
            by_row[m["row"]].append(m)
    GOLD[ds] = {}
    with open(GOLD_OUT/f"{ds}_gold.tsv", "w", encoding="utf-8", newline="") as fh:
        fh.write("row\tid\tn_mentions\tgold_all\tgold_hard\tgold_hard_names\n")
        for row in range(len(CAPS[ds]["expert"])):
            ms = by_row.get(row, [])
            g_all  = sorted({m["chosen"] for m in ms})
            g_hard = sorted({m["chosen"] for m in ms if not m["in_lay"]})
            GOLD[ds][row] = (g_all, g_hard)
            fh.write(f"{row}\t{IDS[ds][row]}\t{len(ms)}\t{';'.join(g_all)}\t{';'.join(g_hard)}\t"
                     f"{' | '.join(name(c) for c in g_hard)}\n")

    n = len(GOLD[ds])
    sizes_all  = [len(a) for a, _ in GOLD[ds].values()]
    sizes_hard = [len(h) for _, h in GOLD[ds].values()]
    amb = [m for m in dec if m["chosen"] and m["rule"] != "unique"]
    SUMMARY.append({
        "dataset": ds, "captions": n,
        "mean_G_all": sum(sizes_all) / n, "mean_G_hard": sum(sizes_hard) / n,
        "captions_G_hard_empty": sum(s == 0 for s in sizes_hard),
        "distinct_cuis_all":  len({c for a, _ in GOLD[ds].values() for c in a}),
        "distinct_cuis_hard": len({c for _, h in GOLD[ds].values() for c in h}),
        "mentions_dropped_by_group": sum(m["rule"] == "dropped_by_group" for m in dec),
        "mentions_in_lay": sum(m["in_lay"] for m in dec),
        **{f"share_{r}": sum(m["rule"] == r for m in amb) / max(len(amb), 1)
           for r in ("group_priority", "preferred_name", "source_count", "lowest_cui")},
    })

summary = pd.DataFrame(SUMMARY)
summary.to_csv(GOLD_OUT/"gold_summary.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
summary.T

In [ ]:
# Cell G10 — STEP 4a: 50 random VALIDATION captions for manual checking. Peak RAM ~0.
# Open the CSV in Excel and fill 'verdict' on every line:
#   ok        = right concept AND right CUI
#   wrong_cui = right concept, wrong CUI (write the right one in 'correct_cui')
#   not_gold  = should not be a gold concept at all
# For every concept the extractor MISSED, add a new line with row, span and verdict = missed.
# This checks Steps 0-2 (gold_all). Step 3 is a fixed string rule and needs no manual check.
DS, N_CHECK, SEED = "mlbv_chest_validation", 50, 42
rows = sorted(random.Random(SEED).sample(range(len(CAPS[DS]["expert"])), N_CHECK))
by_row = defaultdict(list)
for m in DECIDED[DS]:
    if m["chosen"]:
        by_row[m["row"]].append(m)

with open(GOLD_OUT/"manual_check_50.csv", "w", encoding="utf-8-sig", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["row", "id", "expert_caption", "lay_caption", "span", "cui", "name", "groups",
                "rule", "n_candidates", "in_lay", "verdict", "correct_cui", "note"])
    for row in rows:
        for j, m in enumerate(by_row.get(row, [])):
            c = m["chosen"]
            w.writerow([row, IDS[DS][row],
                        TEXT[DS]["expert"][row] if j == 0 else "",
                        TEXT[DS]["lay"][row] if j == 0 else "",
                        m["span"], c, name(c),
                        ",".join(sorted({TUI_GROUP[t] for t in CUI_TUIS[c]})),
                        m["rule"], len(m["cuis_kept"]), m["in_lay"], "", "", ""])
print(f"{sum(len(by_row.get(r, [])) for r in rows)} lines for {N_CHECK} captions -> manual_check_50.csv")

# Small Fix

In [ ]:
# Cell G6 -- STEP 0: greedy longest-match extraction on EXPERT captions. Peak RAM: a few MB.
# Longest match means nested matches are never emitted ("effusion" inside "pleural effusion").
# Single-word code rule: a ONE-word span is skipped if it is shorter than 3 characters or is a
# letter+digit label ('c', 'mo', 'iv', 't1', 's3', 'cm'). These are units, figure/video labels,
# MRI sequences or levels, and their UMLS matches are almost always another sense
# ('c' -> Cockayne Syndrome, 'mo' -> molybdenum). Multi-word spans that contain them still match.
# Exception: 'ct' (computed tomography), the most frequent modality word in the corpus; its sense
# is fixed by the domain-sense table in G8.
SHORT_CODE = re.compile(r"^(?:[a-z]{1,2}|[a-z]{1,3}\d+[a-z]?)$")
SHORT_KEEP = frozenset({"ct"})

def is_short_code(t):
    return bool(SHORT_CODE.match(t)) and t not in SHORT_KEEP

def extract(toks):
    """Left to right; at each position take the longest valid span whose normalized key is in LEX."""
    i, out = 0, []
    while i < len(toks):
        hit = None
        for n in sorted(span_lengths(toks, i), reverse=True):
            if n == 1 and is_short_code(toks[i]):
                continue
            k = nkey(toks[i:i+n])
            if k in LEX:
                hit = (i, n, k)
                break
        if hit:
            out.append(hit)
            i += hit[1]
        else:
            i += 1
    return out

MENTIONS, n_code = {}, Counter()
for ds, d in CAPS.items():
    MENTIONS[ds] = [(row, s, n, " ".join(toks[s:s+n]), k, tuple(sorted(LEX[k])))
                    for row, toks in enumerate(d["expert"]) for s, n, k in extract(toks)]
    n_code[ds] = sum(1 for toks in d["expert"] for t in toks
                     if is_short_code(t) and nkey([t]) in LEX)
    amb = sum(len(m[5]) > 1 for m in MENTIONS[ds])
    print(f"{ds:<24} {len(MENTIONS[ds]):,} mentions | ambiguous {amb:,} ({amb/len(MENTIONS[ds]):.1%}) | "
          f"short-code tokens skipped {n_code[ds]:,}")

In [ ]:
# Cell G7 — STEP 1: which semantic types can be gold. Peak RAM ~0.
# Base: the coverage subtask's CLINICAL_GROUPS minus LIVB, so "gold" stays a subset of "clinical".
GOLD_GROUPS = {
    "DISO": "Disorders: findings and diseases, the main thing a lay reader needs explained",
    "ANAT": "Anatomy: where a finding is",
    "PROC": "Procedures: the imaging study and interventions named in the caption",
    "DEVI": "Devices: tubes, lines and implants visible on the image",
    "CHEM": "Chemicals & Drugs: contrast agents and substances named in the caption",
    "PHYS": "Physiology: body functions and clinical attributes",
    "PHEN": "Phenomena: e.g. test results and biologic processes",
}
# Not gold: CONC (generic ideas: 'left', 'image', 'size'), ACTI, OBJC, OCCU, ORGA, and LIVB, whose
# remaining types after the reference filter are people and groups ('patient', 'man', 'infant').

# Exceptions kept inside a non-gold group
KEEP_TYPES = {
    "T004": "pathogen named as the cause of a finding",   # Fungus (LIVB)
    "T005": "pathogen named as the cause of a finding",   # Virus (LIVB)
    "T007": "pathogen named as the cause of a finding",   # Bacterium (LIVB)
}
# Single types dropped inside gold groups: molecular or research level, never an imaging concept
EXCLUDE_TYPES = {
    "T044": "molecular level",                            # Molecular Function (PHYS)
    "T045": "molecular level",                            # Genetic Function (PHYS)
    "T062": "research, not care",                         # Research Activity (PROC)
    "T063": "molecular research technique",               # Molecular Biology Research Technique (PROC)
}
# Step 2 tie-break order: captions report findings (DISO) located in anatomy (ANAT), then what was done
GROUP_PRIORITY = ["DISO", "ANAT", "PROC", "DEVI", "CHEM", "PHYS", "PHEN", "LIVB"]

# Guards: the policy must be consistent with SemGroups
for t in {**KEEP_TYPES, **EXCLUDE_TYPES}:
    assert t in TUI_GROUP, f"{t} is not in SemGroups"
for t in KEEP_TYPES:
    assert TUI_GROUP[t] not in GOLD_GROUPS, f"{t} is an exception but its group is already gold"
for t in EXCLUDE_TYPES:
    assert TUI_GROUP[t] in GOLD_GROUPS, f"{t} is excluded but its group is not gold anyway"
assert set(GROUP_PRIORITY) >= set(GOLD_GROUPS) | {TUI_GROUP[t] for t in KEEP_TYPES}

def keep_tui(t):
    return t not in EXCLUDE_TYPES and (TUI_GROUP[t] in GOLD_GROUPS or t in KEEP_TYPES)

def keep_cui(c):
    """A CUI can be gold if at least one of its types is allowed."""
    return any(keep_tui(t) for t in CUI_TUIS[c])

for t in sorted({*KEEP_TYPES, *EXCLUDE_TYPES}):
    print(f"  {'keep' if t in KEEP_TYPES else 'drop'}  {t}  {TUI_GROUP[t]}  {TUI_NAME[t]}")

In [ ]:
# Cell G8 -- STEP 2 then STEP 1: decide the sense among ALL candidates, then apply the group filter
# to the winner. Filtering first removed the everyday sense ('left' the side, 'changes' = Change)
# and let a junk concept win ('left:' DISO, 'Menopause'). Peak RAM ~15 MB (mention dicts).
G_RANK = {g: i for i, g in enumerate(GROUP_PRIORITY)}
NOT_GOLD = len(G_RANK)

# Domain-sense table, applied BEFORE the cascade. Selected once from the first G8b audit (40 most
# frequent keys dropped although a gold-eligible sense existed): chest-imaging terms whose cascade
# winner was a generic non-clinical concept, plus 'ct' (kept by G6). Each CUI must be one of the
# key's UMLS candidates and gold-eligible (checked below). The table is frozen after this run.
DOMAIN_SENSE = {nkey([w]): c for w, c in [
    ("consolidation", "C0521530"),  # Lung consolidation -- the airspace finding, not generic 'Consolidation'
    ("contrast",      "C0009924"),  # Contrast Media -- 'with contrast' means the agent, not image contrast
    ("mediastinal",   "C0025066"),  # Mediastinum (T023 Body Part) -- adjective of the anatomy
    ("pulmonary",     "C0024109"),  # Lung (T023) -- adjective of the anatomy
    ("ventricular",   "C0018827"),  # Heart Ventricle (T023) -- adjective of the anatomy
    ("pleural",       "C0032225"),  # Pleura (T023) -- adjective of the anatomy
    ("ct",            "C0040405"),  # X-Ray Computed Tomography (T060) -- the scan, not Carpal Tunnel Syndrome
]}

def rank(c):
    return min((G_RANK[TUI_GROUP[t]] if keep_tui(t) else NOT_GOLD) for t in CUI_TUIS[c])

def choose(k, cands):
    """Domain-sense table first (7 keys), then the cascade:
    unique > UMLS preferred name > most sources (most-frequent-sense proxy) > group priority > lowest CUI."""
    if k in DOMAIN_SENSE and DOMAIN_SENSE[k] in cands:
        return DOMAIN_SENSE[k], "domain_sense"
    if len(cands) == 1:
        return cands[0], "unique"
    p = [c for c in cands if (c, k) in PREF]
    if len(p) == 1:
        return p[0], "preferred_name"
    s = p or list(cands)
    top = max(len(SRC.get((c, k), ())) for c in s)
    s = [c for c in s if len(SRC.get((c, k), ())) == top]
    if len(s) == 1:
        return s[0], "source_count"
    best = min(rank(c) for c in s)
    s = [c for c in s if rank(c) == best]
    if len(s) == 1:
        return s[0], "group_priority"
    return min(s), "lowest_cui"                # reproducibility only; its share should be small

# Guard: every table entry must be a real UMLS candidate of its key and gold-eligible
for k, c in DOMAIN_SENSE.items():
    assert k in LEX and c in LEX[k], f"domain sense {c} is not a UMLS candidate of '{k}'"
    assert keep_cui(c), f"domain sense {c} for '{k}' is not gold-eligible"

DECIDED = {}
for ds, ments in MENTIONS.items():
    out, rules = [], Counter()
    for row, s, n, span, k, cuis in ments:
        winner, rule = choose(k, cuis)
        gold = keep_cui(winner)
        out.append({"row": row, "start": s, "n_words": n, "span": span, "key": k,
                    "cuis_all": cuis, "cuis_kept": [c for c in cuis if keep_cui(c)],
                    "winner": winner, "winner_rule": rule,
                    "chosen": winner if gold else "",
                    "rule": rule if gold else "dropped_by_group"})
        rules[out[-1]["rule"]] += 1
    DECIDED[ds] = out
    print(f"{ds}: " + ", ".join(f"{r} {v:,} ({v/len(ments):.1%})" for r, v in rules.most_common()))

# Audit of ambiguous keys on the VALIDATION split, most frequent first; '*' marks gold-eligible candidates
DS = "mlbv_chest_validation"
freq = Counter(m["key"] for m in DECIDED[DS] if len(m["cuis_all"]) > 1)
first = {}
for m in DECIDED[DS]:
    first.setdefault(m["key"], m)
with open(GOLD_OUT/f"{DS}_disambiguation_audit.csv", "w", encoding="utf-8-sig", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["key", "mentions", "n_candidates", "winner", "winner_name", "winner_rule",
                "gold_kept", "candidates"])
    for k, v in freq.most_common():
        m = first[k]
        cands = " || ".join(
            f"{'*' if keep_cui(c) else ''}{c}: {name(c)} "
            f"[{','.join(sorted({TUI_GROUP[t] for t in CUI_TUIS[c]}))}; "
            f"{len(SRC.get((c, k), ()))} src{'; PREF' if (c, k) in PREF else ''}]"
            for c in m["cuis_all"])
        w.writerow([k, v, len(m["cuis_all"]), m["winner"], name(m["winner"]), m["winner_rule"],
                    bool(m["chosen"]), cands])
print(f"audit: {len(freq):,} ambiguous keys -> {DS}_disambiguation_audit.csv")

In [ ]:
# Cell G8b — words dropped although a medical (gold-eligible) sense existed. Peak RAM ~0.
# If this list is mostly 'left', 'right', 'patient', 'severe' -> correct drops.
# If it shows real findings ('mass', 'opacity', 'lesion') -> the cascade needs one more rule.
DS = "mlbv_chest_validation"
lost, ex = Counter(), {}
for m in DECIDED[DS]:
    if not m["chosen"] and m["cuis_kept"]:
        lost[m["key"]] += 1
        ex.setdefault(m["key"], m)
print(f"mentions dropped although a gold-eligible sense existed: {sum(lost.values()):,} "
      f"({len(lost):,} distinct words)")
for k, v in lost.most_common(40):
    m = ex[k]
    alt = "; ".join(f"{c} {name(c)}" for c in m["cuis_kept"][:2])
    print(f"{v:>5}  {m['span']:<22} winner: {name(m['winner'])[:35]:<35} | medical sense: {alt[:70]}")

In [ ]:
# Cell G9 — STEP 3: drop a term if the lay caption keeps the same normalized term, then build G.
# The comparison is by STRING, not by CUI: 'collapsed lung' is a UMLS name for pneumothorax, so a CUI
# comparison would drop exactly the terms that WERE simplified.
#   gold_all  = CUIs after Steps 0-2
#   gold_hard = gold_all minus CUIs whose every mention also appears verbatim (normalized) in the lay caption
# Peak RAM: a few MB.
def lay_span_keys(toks):
    return {nkey(toks[i:i+n]) for i, n in spans(toks)}

SUMMARY, GOLD = [], {}
for ds, dec in DECIDED.items():
    lay = CAPS[ds]["lay"]
    cur_row, cur_keys = None, set()
    for m in dec:                                    # mentions are in row order
        if m["row"] != cur_row:
            cur_row, cur_keys = m["row"], lay_span_keys(lay[m["row"]])
        m["in_lay"] = bool(m["chosen"]) and m["key"] in cur_keys

    with open(GOLD_OUT/f"{ds}_mentions.tsv", "w", encoding="utf-8", newline="") as fh:
        fh.write("row\tstart\tn_words\tspan\tkey\tcuis_all\tcuis_kept\tchosen\trule\tin_lay\n")
        for m in dec:
            fh.write(f"{m['row']}\t{m['start']}\t{m['n_words']}\t{m['span']}\t{m['key']}\t"
                     f"{';'.join(m['cuis_all'])}\t{';'.join(m['cuis_kept'])}\t{m['chosen']}\t"
                     f"{m['rule']}\t{m['in_lay']}\n")

    by_row = defaultdict(list)
    for m in dec:
        if m["chosen"]:
            by_row[m["row"]].append(m)
    GOLD[ds] = {}
    with open(GOLD_OUT/f"{ds}_gold.tsv", "w", encoding="utf-8", newline="") as fh:
        fh.write("row\tid\tn_mentions\tgold_all\tgold_hard\tgold_hard_names\n")
        for row in range(len(CAPS[ds]["expert"])):
            ms = by_row.get(row, [])
            g_all  = sorted({m["chosen"] for m in ms})
            g_hard = sorted({m["chosen"] for m in ms if not m["in_lay"]})
            GOLD[ds][row] = (g_all, g_hard)
            fh.write(f"{row}\t{IDS[ds][row]}\t{len(ms)}\t{';'.join(g_all)}\t{';'.join(g_hard)}\t"
                     f"{' | '.join(name(c) for c in g_hard)}\n")

    n = len(GOLD[ds])
    sizes_all  = [len(a) for a, _ in GOLD[ds].values()]
    sizes_hard = [len(h) for _, h in GOLD[ds].values()]
    amb = [m for m in dec if m["chosen"] and m["rule"] != "unique"]
    SUMMARY.append({
        "dataset": ds, "captions": n,
        "mean_G_all": sum(sizes_all) / n, "mean_G_hard": sum(sizes_hard) / n,
        "captions_G_hard_empty": sum(s == 0 for s in sizes_hard),
        "distinct_cuis_all":  len({c for a, _ in GOLD[ds].values() for c in a}),
        "distinct_cuis_hard": len({c for _, h in GOLD[ds].values() for c in h}),
        "mentions_dropped_by_group": sum(m["rule"] == "dropped_by_group" for m in dec),
        "mentions_in_lay": sum(m["in_lay"] for m in dec),
        **{f"share_{r}": sum(m["rule"] == r for m in amb) / max(len(amb), 1)
           for r in ("group_priority", "preferred_name", "source_count", "lowest_cui")},
    })

summary = pd.DataFrame(SUMMARY)
summary.to_csv(GOLD_OUT/"gold_summary.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
summary.T

In [ ]:
# Cell G10 — STEP 4a: 50 random VALIDATION captions for manual checking. Peak RAM ~0.
# Open the CSV in Excel and fill 'verdict' on every line:
#   ok        = right concept AND right CUI
#   wrong_cui = right concept, wrong CUI (write the right one in 'correct_cui')
#   not_gold  = should not be a gold concept at all
# For every concept the extractor MISSED, add a new line with row, span and verdict = missed.
# This checks Steps 0-2 (gold_all). Step 3 is a fixed string rule and needs no manual check.
DS, N_CHECK, SEED = "mlbv_chest_validation", 50, 42
rows = sorted(random.Random(SEED).sample(range(len(CAPS[DS]["expert"])), N_CHECK))
by_row = defaultdict(list)
for m in DECIDED[DS]:
    if m["chosen"]:
        by_row[m["row"]].append(m)

with open(GOLD_OUT/"manual_check_50.csv", "w", encoding="utf-8-sig", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["row", "id", "expert_caption", "lay_caption", "span", "cui", "name", "groups",
                "rule", "n_candidates", "in_lay", "verdict", "correct_cui", "note"])
    for row in rows:
        for j, m in enumerate(by_row.get(row, [])):
            c = m["chosen"]
            w.writerow([row, IDS[DS][row],
                        TEXT[DS]["expert"][row] if j == 0 else "",
                        TEXT[DS]["lay"][row] if j == 0 else "",
                        m["span"], c, name(c),
                        ",".join(sorted({TUI_GROUP[t] for t in CUI_TUIS[c]})),
                        m["rule"], len(m["cuis_kept"]), m["in_lay"], "", "", ""])
print(f"{sum(len(by_row.get(r, [])) for r in rows)} lines for {N_CHECK} captions -> manual_check_50.csv")

In [ ]:
# Cell G10b -- copy the draft verdicts into the fresh manual_check_50.csv, matched by (row, span).
# The draft was made on the 4th attempt and is archived in try04/.
# 'missed' lines are kept only if the extractor still does not produce that span for that row.
# Lines new in this run (e.g. one-word 'ct') come out with an empty verdict for you to fill. Peak RAM ~0.
DRAFT_PATH = GOLD_OUT/"try04"/"manual_check_50_draft.csv"
assert DRAFT_PATH.exists(), f"draft not found: {DRAFT_PATH}"

draft = pd.read_csv(DRAFT_PATH, dtype=str, encoding="utf-8-sig").fillna("")
new = pd.read_csv(GOLD_OUT/"manual_check_50.csv", dtype=str, encoding="utf-8-sig").fillna("")
look = {(r.row, r.span): (r.verdict, r.correct_cui, r.note)
        for r in draft.itertuples() if r.verdict != "missed"}
got = [look.get((r, s), ("", "", "")) for r, s in zip(new["row"], new["span"])]
new["verdict"], new["correct_cui"], new["note"] = zip(*got)

have = set(zip(new["row"], new["span"]))
keep = [(v == "missed") and ((r, s) not in have) for r, s, v in zip(draft["row"], draft["span"], draft["verdict"])]
missed = draft[keep]
new["_o"], missed = range(len(new)), missed.assign(_o=10**6)
out = (pd.concat([new, missed], ignore_index=True)
         .assign(_r=lambda d: d["row"].astype(int))
         .sort_values(["_r", "_o"], kind="stable").drop(columns=["_r", "_o"]))
out.to_csv(GOLD_OUT/"manual_check_50.csv", index=False, encoding="utf-8-sig")
print(f"{len(out)} lines | draft verdicts copied {sum(v != '' for v in new['verdict'])} | "
      f"missed lines kept {len(missed)} | lines still blank {(out['verdict'] == '').sum()}")

In [ ]:
# Back up the pre-filled manual check so a G10 re-run can't wipe it. Peak RAM ~0.
import shutil
(GOLD_OUT/"final").mkdir(exist_ok=True)
shutil.copy2(GOLD_OUT/"manual_check_50.csv", GOLD_OUT/"final"/"manual_check_50_prefilled.csv")
print("saved ->", GOLD_OUT/"final"/"manual_check_50_prefilled.csv")

# Rerun coverage

In [ ]:
# Cell C1 -- KB coverage re-run with the GOLD extractor: span rules (G4), short-code rule (G6),
# domain-sense table (G8). Reuses CAPS/MENTIONS/DECIDED/CUI_TUIS in this kernel; reads no UMLS file.
# Peak RAM ~0.
COV_OUT = Path(r"D:\Project\Radiology_RAG\data\subtask_data_coverage\coverage_results")
COV_OUT.mkdir(parents=True, exist_ok=True)
SUFFIX_V2 = "_v2_goldextractor"            # old coverage files (the 65.2% run) stay untouched

# Same clinical definition as coverage notebook Cell 13
CLINICAL_GROUPS = {"ANAT", "DISO", "PROC", "DEVI", "CHEM", "PHYS", "PHEN", "LIVB"}

def groups(c):
    return {TUI_GROUP[t] for t in CUI_TUIS[c]}

def is_content(t):
    """Content word = not the sentence-boundary token, not a stopword, not a pure number."""
    return t != BOUNDARY and t not in ENGLISH_STOP_WORDS and not t.isdigit()

assert all(len(MENTIONS[ds]) == len(DECIDED[ds]) for ds in MENTIONS), "re-run G6 -> G8 first"
print("ready:", {ds: f"{len(DECIDED[ds]):,} mentions" for ds in DECIDED})

In [ ]:
# Cell C2 -- token coverage summary. Peak RAM ~0.
# token_cov_clinical_candidate = OLD definition (span counts if ANY candidate CUI is clinical);
#                                compare this one with the old 0.652.
# token_cov_clinical_decided   = NEW definition (span counts only if the sense chosen in G8 is clinical);
#                                this is the figure consistent with the gold concepts.
rows = []
for ds, d in CAPS.items():
    by_row = defaultdict(list)
    for m in DECIDED[ds]:
        by_row[m["row"]].append(m)
    tot = cov_any = cov_cand = cov_dec = 0
    D, D_cand, D_dec, D_strict = set(), set(), set(), set()
    for row, toks in enumerate(d["expert"]):
        in_any, in_cand, in_dec = set(), set(), set()
        for m in by_row.get(row, []):
            span = range(m["start"], m["start"] + m["n_words"])
            cuis = m["cuis_all"]
            in_any.update(span)
            D.update(cuis)
            if len(cuis) == 1:
                D_strict.add(cuis[0])
            clin = [c for c in cuis if groups(c) & CLINICAL_GROUPS]
            if clin:
                in_cand.update(span)
                D_cand.update(clin)
            if groups(m["winner"]) & CLINICAL_GROUPS:
                in_dec.update(span)
                D_dec.add(m["winner"])
        for i, t in enumerate(toks):
            if is_content(t):
                tot += 1
                cov_any  += i in in_any
                cov_cand += i in in_cand
                cov_dec  += i in in_dec
    rows.append({"dataset": ds, "captions": len(d["expert"]), "content_tokens": tot,
                 "token_cov_any_match": cov_any / tot,
                 "token_cov_clinical_candidate": cov_cand / tot,
                 "token_cov_clinical_decided": cov_dec / tot,
                 "concepts_D": len(D), "concepts_D_clinical": len(D_cand),
                 "concepts_D_decided_clinical": len(D_dec),
                 "concepts_D_unambiguous": len(D_strict)})
cov2 = pd.DataFrame(rows)
cov2.to_csv(COV_OUT/f"coverage_summary{SUFFIX_V2}.csv", index=False)
pd.options.display.float_format = "{:.3f}".format
cov2

In [ ]:
# Cell C3 -- concept set D (validation) saved for the KB step, plus D by semantic group.
# 'winner_mentions' = how often G8 chose this CUI as the sense; the by-group table counts winners only.
# Peak RAM ~0.
DS = "mlbv_chest_validation"
freq = Counter(c for m in DECIDED[DS] for c in m["cuis_all"])
wins = Counter(m["winner"] for m in DECIDED[DS])

with open(COV_OUT/f"{DS}_concepts_D{SUFFIX_V2}.tsv", "w", encoding="utf-8", newline="") as fh:
    fh.write("CUI\tname\tmentions\twinner_mentions\tgroups\tclinical\n")
    for c, v in freq.most_common():
        g = groups(c)
        fh.write(f"{c}\t{name(c)}\t{v}\t{wins[c]}\t{','.join(sorted(g))}\t{bool(g & CLINICAL_GROUPS)}\n")

by_group = []
for g in sorted(GROUP_NAME):
    Wg = [c for c in wins if g in groups(c)]
    if Wg:
        by_group.append({"group": g, "name": GROUP_NAME[g], "clinical": g in CLINICAL_GROUPS,
                         "concepts_chosen": len(Wg), "mentions": sum(wins[c] for c in Wg)})
grp2 = pd.DataFrame(by_group).sort_values("concepts_chosen", ascending=False)
grp2.to_csv(COV_OUT/f"{DS}_D_by_group{SUFFIX_V2}.csv", index=False)
print(f"saved {len(freq):,} candidate CUIs ({len(wins):,} chosen as a sense) -> {DS}_concepts_D{SUFFIX_V2}.tsv")
grp2

In [ ]:
# Cell G10c -- fill the one remaining blank verdict (row 814, 'ct' = X-Ray Computed Tomography, correct). Peak RAM ~0.
from pathlib import Path
import shutil
import pandas as pd

GOLD_OUT = Path(r"D:\Project\Radiology_RAG\data\subtask_gold_concepts_det")
f = GOLD_OUT/"manual_check_50.csv"
ann = pd.read_csv(f, dtype=str, encoding="utf-8-sig").fillna("")

sel = (ann["row"] == "814") & (ann["span"] == "ct") & (ann["verdict"] == "")
assert sel.sum() == 1, f"expected 1 blank 'ct' line in row 814, found {sel.sum()}"
ann.loc[sel, "verdict"] = "ok"
ann.loc[sel, "note"] = "one-word code added in the 5th run; domain-sense key"

blank = (ann["verdict"].str.strip() == "").sum()
assert blank == 0, f"{blank} lines still have no verdict"
ann.to_csv(f, index=False, encoding="utf-8-sig")
(GOLD_OUT/"final").mkdir(exist_ok=True)
shutil.copy2(f, GOLD_OUT/"final"/"manual_check_50_final.csv")
print(f"{len(ann)} lines, all with a verdict -> manual_check_50.csv + final/manual_check_50_final.csv")

In [ ]:
# Cell G11 -- score the filled manual check. Mention-level. Peak RAM ~0.
#   TP = ok | FP = wrong_cui + not_gold | FN = wrong_cui + missed  (a wrong CUI also misses the right one)
import json
VALID = {"ok", "wrong_cui", "not_gold", "missed"}
ann = pd.read_csv(GOLD_OUT/"manual_check_50.csv", dtype=str, encoding="utf-8-sig").fillna("")
ann["v"] = ann["verdict"].str.strip().str.lower()

typos = sorted(set(ann["v"]) - VALID)
assert not typos, f"unknown or empty verdicts: {typos}"
print(f"captions: {ann['row'].nunique()} | lines: {len(ann)}")

cnt = ann["v"].value_counts()
tp, wrong = int(cnt.get("ok", 0)), int(cnt.get("wrong_cui", 0))
fp, fn = wrong + int(cnt.get("not_gold", 0)), wrong + int(cnt.get("missed", 0))
P, R = tp / (tp + fp), tp / (tp + fn)
F1 = 2 * P * R / (P + R)
print(f"precision {P:.3f} | recall {R:.3f} | F1 {F1:.3f} | counts {dict(cnt)}")

# Disambiguation accuracy per rule: among lines whose concept was right, was the CUI right?
dis = ann[ann["v"].isin(["ok", "wrong_cui"])]
display(dis.groupby("rule")["v"].agg(lines="size", accuracy=lambda s: round((s == "ok").mean(), 3)))

# Which rule produced the not_gold lines
display(ann[ann["v"] == "not_gold"].groupby("rule").size().rename("not_gold_lines"))

(GOLD_OUT/"final"/"manual_check_scores.json").write_text(json.dumps(
    {"captions": int(ann["row"].nunique()), "lines": len(ann), "counts": {k: int(v) for k, v in cnt.items()},
     "precision": round(P, 4), "recall": round(R, 4), "f1": round(F1, 4),
     "annotation": "LLM pre-annotation, reviewed by the author"}, indent=2), encoding="utf-8")
print("saved -> final/manual_check_scores.json")